# **Introduction Deep Learning**

In [ ]:
######## PACOTES NECESSÁRIOS ##########

import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

Vamos criar a pasta `images/ann` (caso ainda não exista) e definir a função `save_fig()`, que será usada neste notebook para salvar as figuras em alta resolução:

In [ ]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "ann"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

## **1 - Regressão**

Vamos importar o `dataset` a partir do `sklearn`, usaremos o famoso banco de dados `housing`:

In [ ]:
# Preparação dos Dados (Apenas extração e padronização para a Parte 1)
california = fetch_california_housing()
X, y = california.data, california.target

print("Nome da variável alvo:", california.target_names)
print("Nome das variáveis explicativas:", california.feature_names)

print("Dimensão dos dados:", X.shape)

Faremos algumas manipulações noa dados antes de os passarmos ao modelo, chamamos essa fase/etapa de **pré-processamento de dados**:

In [ ]:
# É sempre importante randomizar os dados antes de treinar o modelo, para evitar qualquer viés na ordem dos dados
np.random.seed(42)  # Para reprodutibilidade
indices = np.random.permutation(X.shape[0])
X = X[indices]
y = y[indices]

# Essencial para o Gradiente Descendente (SGD) convergir na regressão
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Para não sobrecarregar o tempo de aula, vamos treinar com um subconjunto
X_train_demo = X_scaled[:2000]
y_train_demo = y[:2000]

Vamos importar o método `Sequential` da API [Keras](https://www.tensorflow.org/guide/keras?hl=pt-br). O ***Sequential-API*** é o tipo mais simples de método do **Keras** para redes neurais constituídas somente de uma única pilha de camadas sequencialmente conectadas

A função `Dense` (Dense Layer) define uma camada de rede neural onde cada neurônio da camada está conectado a cada neurônio da camada anterior. É o tipo de camada mais comum em redes neurais e é usada para aprender padrões complexos nos dados.

A função `Input` (Input Layer) define a forma dos dados de entrada que serão passados para a rede neural. Ela não realiza nenhuma transformação nos dados, apenas especifica o formato esperado.

Veremos abaixo como usar essas funções:

:

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# Construindo a Arquitetura (Sequential API)
model = Sequential([
    Input(shape=(X_train_demo.shape[1],)),  # 8 features de entrada
    Dense(units=3, activation='relu', name='Camada_Oculta_1', use_bias=True),
    Dense(units=2, activation='tanh', name='Camada_Oculta_2', use_bias=True),
    #Dense(units=7, activation='relu', name='Camada_Oculta_3', use_bias=True),
    #Dense(units=4, activation='tanh', name='Camada_Oculta_4', use_bias=True),
    Dense(units=1, activation='linear', name='Camada_Saida_Regressao', use_bias=True)
])

model.summary()

In [ ]:
# código para mostrar em imagem de tabela a arquitetura da rede neural
tf.keras.utils.plot_model(model,
                          "my_housing_model.png",
                          show_shapes=True,           # Mostra as formas dos tensores em cada camada
                          show_layer_names=True,      # Mostra o nome que você deu para a camada
                          show_layer_activations=True # Mostra a função de ativação (ReLU, Linear, etc)
                          )

In [ ]:
model.layers

In [ ]:
hidden0 = model.layers[0]
hidden0.name

In [ ]:
model.get_layer('Camada_Oculta_1') is hidden0

In [ ]:
weights_0, biases_0 = hidden0.get_weights()
print("Pesos:\n", weights_0)
print("\nBias:\n", biases_0)

Da forma como o modelo foi criado acima, os inicializadores dos parâmetros foram gerados pelo inicializador `glorot_uniform`, que é a configuração *default* do método *Sequential-API*, se quisessemos alterar esses inicializadores teriamos que configurar a funnção `kernel_initializer`, como no exemplo abaixo:

In [ ]:
from tensorflow.keras import initializers, regularizers

model_inic = Sequential([
    Input(shape=(X_train_demo.shape[1],)),

    # Controle completo dos parâmetros
    Dense(units=3, activation='relu', name='Camada_Oculta_1',
          kernel_initializer=initializers.HeNormal(seed=42),  # Seed para reprodutibilidade
          bias_initializer=initializers.Zeros()),

    Dense(units=2, activation='relu', name='Camada_Oculta_2',
          kernel_initializer=initializers.GlorotUniform(seed=123),
          bias_initializer=initializers.Ones()),

    Dense(units=1, activation='linear', name='Camada_Saida_Regressao',
          kernel_initializer=initializers.RandomNormal(mean=5, stddev=1, seed=456),
          bias_initializer=initializers.Constant(value=0.5))
])

model_inic.summary()

In [ ]:
hidden0 = model_inic.layers[2]
weights_0, biases_0 = hidden0.get_weights()
print("Pesos:\n", weights_0)
print("\nBias:\n", biases_0)

Existem os principais inicializadores:
 | Inicializador | Descrição | Uso típico |
|--------------|-----------|------------|
| `'zeros'` | Todos os pesos = 0 | Bias apenas |
| `'ones'` | Todos os pesos = 1 | Bias apenas |
| `'random_normal'` | Distribuição normal | Geral |
| `'random_uniform'` | Distribuição uniforme | Geral |
| `'glorot_uniform'` | Xavier uniforme (padrão) | Tanh, Sigmoid |
| `'glorot_normal'` | Xavier normal | Tanh, Sigmoid |
| `'he_uniform'` | He uniforme | ReLU |
| `'he_normal'` | He normal | ReLU |
| `'lecun_uniform'` | LeCun uniforme | SELU |

Se quisermos visualizar a estrutura do nosso modelo de rede neural, podemos usar grafos para ter uma noção visual da estrutura do nosso modelo, usando a função `plot_neural_graph()` que desenvolvi para esse documento:

In [ ]:
from neural_graph import plot_neural_graph

plot_neural_graph(model)
save_fig('graph_neuron')

E se quisermos ir mais **FUNDO!** Podemos ver a estrutura matemática (hipótese) do nosso modelo, usando a função `model_to_math_markdown` que também desenvolvi oara este documento:

In [ ]:
from MathRender import model_to_math_markdown

# Visualização antes do treinamento (Pesos Aleatórios Iniciais)
display(Markdown("### Condição Inicial (Antes do Treinamento)"))
display(Markdown(model_to_math_markdown(model, precision=2)))

Após a criação de um modelo, devemos chamar seu método `compile()` para especificar a função de perda e o otimizador a ser usado, Como opção, podemos especificar uma lista de métricas extras a serem calculadas durante o treinamento e a avaliação:

In [ ]:
# Regressão: Custo é o Erro Quadrático Médio (MSE), Otimizador SGD
model.compile(optimizer="sgd", loss='mse', metrics=["mae"])

print(f"Otimizador configurado: {model.optimizer.get_config()}")
print(f"Função de perda configurada: {model.loss}")
print(f"Métricas configuradas: {[m.name for m in model.metrics]}")

Ao usar o otimizador `SGD`, é importante acertar a taxa de aprendizado. Geralmente, queremos usar o `optimizer = tensorflow.keras.optimizers.SGD(lr=???)` para definir a taxa de aprendizado, em vez de `optimizer = "sgd"`, cujo padrão é `lr=0.01`:

In [ ]:
from tensorflow.keras.optimizers import SGD

opt = SGD(learning_rate=0.05)
model.compile(optimizer=opt, loss='mse', metrics=["mae"])

print(f"Otimizador configurado: {model.optimizer.get_config()}")
print(f"Função de perda configurada: {model.loss}")
print(f"Métricas configuradas: {[m.name for m in model.metrics]}")

### **Erro Quadrático Médio (MSE - Mean Squared Error)**

O Erro Quadrático Médio (MSE) é uma das métricas mais comuns para avaliar modelos de regressão. Ele calcula a média dos quadrados das diferenças entre os valores previstos (`ŷᵢ`) e os valores reais (`yᵢ`).

**Fórmula:**
$$ MSE = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2 $$

**Características:**
*   **Penaliza erros maiores:** Devido ao termo ao quadrado, o MSE penaliza erros maiores de forma mais significativa do que erros menores. Isso significa que ele é mais sensível a *outliers*.
*   **Unidades:** As unidades do MSE são o quadrado das unidades da variável alvo, o que pode dificultar a interpretação direta.
*   **Derivável:** É uma função contínua e derivável, o que a torna adequada para otimizadores baseados em gradiente (como o SGD).

### **Erro Absoluto Médio (MAE - Mean Absolute Error)**

O Erro Absoluto Médio (MAE) é outra métrica popular para regressão. Ele calcula a média dos valores absolutos das diferenças entre os valores previstos (`ŷᵢ`) e os valores reais (`yᵢ`).

**Fórmula:**
$$ MAE = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i| $$

**Características:**
*   **Menos sensível a *outliers*:** Ao usar o valor absoluto, o MAE dá a mesma importância a todos os erros, independentemente de seu tamanho. Isso o torna mais robusto a *outliers* em comparação com o MSE.
*   **Unidades interpretáveis:** As unidades do MAE são as mesmas da variável alvo, o que o torna mais fácil de interpretar diretamente (por exemplo, "o erro médio é de X dólares").
*   **Não derivável em 0:** A função de valor absoluto não é suavemente derivável em zero, o que pode ser um pequeno desafio para alguns algoritmos de otimização, embora na prática isso raramente seja um problema para redes neurais.

In [ ]:
from MathRender import MathRenderCallback

# Treinamento com o Callback de Impressão Matemática
history_1 = model.fit(
    X_train_demo,
    y_train_demo,
    epochs=50,
    batch_size=32,
    verbose=0, # Desligamos a barra de progresso padrão
    callbacks=[MathRenderCallback(print_every_n_epochs=10, precision=2)]
)

In [ ]:
# Treinamento com o Callback de Impressão Matemática
history_2 = model.fit(
    X_train_demo,
    y_train_demo,
    epochs=50,
    batch_size=32,
    verbose=1,
)

In [ ]:
pd.DataFrame(history_2.history).plot(
    figsize=(8, 5), grid=True, xlabel="Epoch",
    style=["r--", "r--.", "b-", "b-*"])
plt.legend(loc="lower left")  # extra code
save_fig("keras_learning_curves_regression")  # extra code
plt.show()

O parâmetro `batch_size` em `model.fit()` define o número de amostras de treinamento que serão propagadas através da rede neural antes que os parâmetros do modelo (pesos e vieses) sejam atualizados.

Por exemplo, se você tem 2000 amostras de treinamento (`X_train_demo`) e um `batch_size` de 32, o modelo processará 32 amostras, calculará o gradiente da função de perda com base nessas 32 amostras, e então ajustará seus pesos. Esse processo se repete até que todas as 2000 amostras tenham sido usadas para uma atualização. **É importante notar que, para cada época, o conjunto de dados é geralmente embaralhado e as amostras são selecionadas sem reposição, garantindo que cada amostra seja usada exatamente uma vez em um lote por época.** Um ciclo completo por todas as amostras de treinamento é chamado de **época** (`epochs`).

## **2 - Classificação**

In [ ]:
from sklearn.datasets import load_breast_cancer

In [ ]:
cancer_data = load_breast_cancer()

In [ ]:
#Preparação dos Dados (Classificação: Benigno vs Maligno)
cancer_data = load_breast_cancer()
X_class = cancer_data.data
y_class = cancer_data.target # 0 (Maligno) ou 1 (Benigno)

print("Nome da variável alvo:", cancer_data.target_names)
print("Nome das variáveis explicativas:", cancer_data.feature_names)

print("Dimensão dos dados:", X_class.shape)

In [ ]:
# É sempre importante randomizar os dados antes de treinar o modelo, para evitar qualquer viés na ordem dos dados
np.random.seed(42)  # Para reprodutibilidade
indices = np.random.permutation(X_class.shape[0])
X_class = X_class[indices]
y_class = y_class[indices]

# Função de Padronização (Ainda mais crucial em classificação para a Sigmoide não saturar!)
scaler_class = StandardScaler()
X_class_scaled = scaler_class.fit_transform(X_class)

# Pegando um subconjunto para o treinamento rápido na aula
X_train_class = X_class_scaled[:400]
y_train_class = y_class[:400]

In [ ]:
# Construindo a Arquitetura
model_class = Sequential([
    Input(shape=(X_train_class.shape[1],)), # 30 variáveis médicas de entrada
    Dense(units=10, activation='linear', name='Oculta_1'),
    Dense(units=6, activation='tanh', name='Oculta_2'),
    Dense(units=4, activation='relu', name='Oculta_3'),
    # A GRANDE MUDANÇA: 1 neurônio na saída, mas com ativação SIGMOID
    Dense(units=1, activation='sigmoid', name='Probabilidade')
])

model_class.summary()

In [ ]:
# código para mostrar em imagem de tabela a arquitetura da rede neural
tf.keras.utils.plot_model(model_class,
                          "cancer.png",
                          show_shapes=True,           # Mostra as formas dos tensores em cada camada
                          show_layer_names=True,      # Mostra o nome que você deu para a camada
                          show_layer_activations=True # Mostra a função de ativação (ReLU, Linear, etc)
                          )

In [ ]:
# Visualizando a Arquitetura e a Matemática inicial
print(">>> Grafo da Rede Neural (Classificação) <<<")
plot_neural_graph(model_class, max_neurons_per_layer=6) # Reutilizando a função que criamos!

In [ ]:
print("\n>>> Equação Matemática Inicial <<<")
display(Markdown(model_to_math_markdown(model_class, precision=2)))

In [ ]:
# 4. Compilação (Nova Função de Custo)
# Como a saída é probabilidade (0 a 1), não usamos MSE. Usamos Entropia Cruzada Binária.
model_class.compile(optimizer="sgd",
                    loss='binary_crossentropy',
                    metrics=['accuracy']) # Adicionamos a acurácia para ver a % de acerto

### Entropia Cruzada Binária (Log-Loss)

A Entropia Cruzada Binária, frequentemente chamada de Log-Loss, é a função de perda padrão para problemas de classificação binária (onde há apenas duas classes, como 0 ou 1, benigno ou maligno). Ela mede o desempenho de um modelo de classificação cuja saída é um valor de probabilidade entre 0 e 1.

**Fórmula:**
$$ L = -\frac{1}{N} \sum_{i=1}^{N} [y_i \log(p_i) + (1 - y_i) \log(1 - p_i)] $$
Onde:
*   `N` é o número de amostras.
*   `yᵢ` é o rótulo verdadeiro da amostra `i` (0 ou 1).
*   `pᵢ` é a probabilidade prevista pelo modelo para a amostra `i` pertencer à classe 1.

**Características:**
*   **Penaliza fortemente previsões incorretas com alta confiança:** Se o modelo prevê uma alta probabilidade para a classe errada, a penalidade (perda) é muito alta. Por exemplo, se `yᵢ=1` mas `pᵢ` está próximo de 0, `log(pᵢ)` se aproxima de `-infinito`, resultando em uma perda muito grande.
*   **Adequada para probabilidades:** Projetada para otimizar modelos que produzem saídas probabilísticas.
*   **Derivável:** É uma função contínua e derivável, o que a torna adequada para otimizadores baseados em gradiente.

### Acurácia (Accuracy)

A Acurácia é uma métrica de avaliação simples e intuitiva, especialmente para problemas de classificação. Ela mede a proporção de previsões corretas (tanto positivos verdadeiros quanto negativos verdadeiros) em relação ao número total de previsões.

**Fórmula:**
$$ Acurácia = \frac{\text{Número de previsões corretas}}{\text{Número total de previsões}} $$

**Características:**
*   **Fácil interpretação:** Representa a porcentagem de instâncias classificadas corretamente.
*   **Bom para datasets balanceados:** Funciona bem quando o número de amostras em cada classe é aproximadamente igual. Em datasets desbalanceados, uma alta acurácia pode ser enganosa (por exemplo, um modelo que prevê sempre a classe majoritária terá alta acurácia, mas não será útil).
*   **Não sensível à magnitude do erro:** Diferente das funções de perda, a acurácia apenas diz se a previsão estava certa ou errada, sem considerar o quão "certa" ou "errada" ela foi (por exemplo, uma previsão de 0.51 para `y=1` é tratada da mesma forma que uma previsão de 0.99 para `y=1`).

In [ ]:
# 5. Treinamento com o nosso Callback
print("Iniciando Treinamento...\n")
history_class_1 = model_class.fit(
    X_train_class,
    y_train_class,
    epochs=100,
    batch_size=32,
    verbose=0,
    callbacks=[MathRenderCallback(print_every_n_epochs=20, precision=2)]
)

In [ ]:
# 5. Treinamento com o nosso Callback
print("Iniciando Treinamento...\n")
history_class_2 = model_class.fit(
    X_train_class,
    y_train_class,
    epochs=100,
    batch_size=32,
)

In [ ]:
pd.DataFrame(history_class_2.history).plot(
    figsize=(8, 5), grid=True, xlabel="Epoch",
    style=["r--", "r--.", "b-", "b-*"])
plt.legend(loc="center left")  # extra code
save_fig("keras_learning_curves_classification")  # extra code
plt.show()

## **3 - Treino, Validação e Teste**

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

In [ ]:
# Carregando os Dados
california = fetch_california_housing()
X, y = california.data, california.target

# A Grande Divisão (Train, Test)
# Primeiro, separamos o que a rede NUNCA vai ver durante o treino (20% para Teste)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# ATENÇÃO: O fit (cálculo da média e desvio) é feito APENAS no treino!
# O teste usa a mesma régua do treino para evitar vazamento de dados (data leakage).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

print(f"Dados de Treino/Validação: {X_train_scaled.shape[0]} amostras")
print(f"Dados de Teste (Invisíveis): {X_test_scaled.shape[0]} amostras\n")

In [ ]:
# Aqui podemos usar o otimizador Adam (mais moderno que o SGD) e mais neurônios
model_real = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu', name='Oculta_1'),
    Dense(32, activation='relu', name='Oculta_2'),
    Dense(1, activation='linear', name='Saida_Preco')
])

In [ ]:
model_real.summary()

### **Função de Perda RMSE Personalizada**

O Keras, por padrão, não oferece 'rmse' (Root Mean Squared Error) como um identificador de função de perda diretamente como 'mse' ou 'mae'. No entanto, o RMSE é a raiz quadrada do MSE.

Para utilizar o RMSE como função de perda, podemos definir uma função personalizada em Keras que calcula a raiz quadrada do Mean Squared Error. Isso garante que o otimizador minimize diretamente o RMSE.

**Por que MSE (ou MAE) é mais comum como Loss Function?**

Embora o RMSE seja uma métrica de avaliação muito intuitiva, o MSE (Mean Squared Error) é mais comumente usado como *função de perda* em redes neurais de regressão por algumas razões:

1.  **Suavidade e Diferenciabilidade:** A função MSE é suave e continuamente diferenciável em toda a sua extensão, o que é ideal para otimizadores baseados em gradiente (como SGD, Adam, etc.). A raiz quadrada, quando aplicada ao MSE, pode introduzir problemas de diferenciabilidade em zero para gradientes, embora na prática isso seja mitigado pelo fato de o erro nunca ser exatamente zero.
2.  **Propriedades Matemáticas:** O MSE tem propriedades matemáticas desejáveis que facilitam sua otimização. Ele amplifica os erros maiores, incentivando o modelo a corrigi-los mais agressivamente.

No entanto, nosso objetivo principal é otimizar diretamente para o RMSE, a criação de uma função de perda personalizada é a abordagem correta.

In [ ]:
from tensorflow.keras import backend as K

def rmse_loss(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

# Recompilando o modelo com a função de perda RMSE personalizada
model_real.compile(optimizer=Adam(learning_rate=0.005), loss=rmse_loss, metrics=['mae'])

print(f"Otimizador configurado: {model_real.optimizer.get_config()}")
print(f"Função de perda configurada: {model_real.loss}")
print(f"Métricas configuradas: {[m.name for m in model_real.metrics]}")

### **Amostragem para Treino, Validação e Teste**

Ao dividir seus dados para o treinamento de modelos de Machine Learning e Deep Learning, é crucial entender como a amostragem é realizada e o papel de cada conjunto de dados.

#### **1. Divisão Inicial (Treino/Validação vs. Teste)**

A primeira etapa é sempre separar um conjunto de **Teste** (`X_test`, `y_test`). Este conjunto de dados deve ser **totalmente invisível** para o modelo durante todo o processo de treinamento e ajuste de hiperparâmetros. Ele serve para fornecer uma avaliação **imparcial** do desempenho final do modelo. O objetivo é simular como o modelo se comportará com dados **novos e não vistos**.

*   **Amostragem Aleatória Simples:** A divisão inicial para o conjunto de teste é feita usando **amostragem aleatória simples**. Funções como `train_test_split` do scikit-learn realizam isso por padrão. Isso significa que cada amostra nos dados originais tem a mesma chance de ser selecionada para o conjunto de teste.
*   **Sem Reposição:** Essa amostragem é feita **sem reposição**. Uma vez que uma amostra é atribuída ao conjunto de treino/validação ou teste, ela não pode ser atribuída a outro. Isso garante que não haja sobreposição entre os conjuntos e que a avaliação seja justa.

#### **2. Divisão do Conjunto de Treino (Treino vs. Validação)**

Após separar o conjunto de teste, o restante dos dados é dividido em **Treino** (`X_train_full`, `y_train_full`) e **Validação**.

*   **Conjunto de Treino:** Usado para **ajustar os pesos** (parâmetros internos) do modelo durante o processo de retropropagação. O modelo *aprende* com esses dados.
*   **Conjunto de Validação:** Usado para **monitorar o desempenho do modelo** durante o treinamento e **ajustar hiperparâmetros** (como taxa de aprendizado, número de camadas, número de neurônios, _batch size_, etc.) sem tocar no conjunto de teste. Ele ajuda a identificar *overfitting* (sobreajuste).

Quando você usa `validation_split` dentro do método `model.fit()` do Keras (como no exemplo `history_real = model_real.fit(..., validation_split=0.2)`):

*   Uma porcentagem especificada (e.g., 20%) do `X_train_full` é **separada aleatoriamente** no início do treinamento para ser o conjunto de validação.
*   Esta amostragem também é **sem reposição** entre os subconjuntos de treino e validação dentro daquele `fit` específico. As amostras de validação são usadas apenas para calcular a função de perda e métricas ao final de cada época, mas **não são usadas para atualizar os pesos do modelo**.

#### **3. Interação entre Validação e Teste**

É **fundamental** que:

*   Os dados de **validação e teste sejam completamente independentes** e não se sobreponham. **Dados do conjunto de validação NUNCA devem ir para o conjunto de teste**, e vice-versa.
*   A divisão em `validation_split` é feita *dentro* do conjunto de treinamento disponível (o `X_train_full` que sobrou após a separação do teste).
*   **Mudança de Época:** A cada época, o modelo é treinado no conjunto de treino e avaliado no conjunto de validação. **As amostras do conjunto de validação permanecem as mesmas** ao longo de todas as épocas para um dado `model.fit()`.

**Em resumo:** A amostragem é aleatória e sem reposição para garantir que os conjuntos de dados (treino, validação, teste) sejam distintos e independentes. O conjunto de teste é mantido separado para uma avaliação final e imparcial, enquanto o conjunto de validação ajuda a otimizar o modelo durante o desenvolvimento.

In [ ]:
# O Treinamento com Validação (Onde a mágica do monitoramento acontece)
print(">>> INICIANDO TREINAMENTO (Monitorando Validação) <<<")
# O validation_split=0.2 tira 20% do X_train_scaled apenas para validar a cada época!
history_real = model_real.fit(
    X_train_scaled,
    y_train_full,
    epochs=60,
    batch_size=64,
    validation_split=0.2,
)

In [ ]:
# Visualizando as Curvas de Aprendizado (O momento de explicar Overfitting)
plt.figure(figsize=(10, 6))
plt.plot(history_real.history['loss'], label='Erro no Treino (Loss)', linewidth=2)
plt.plot(history_real.history['val_loss'], label='Erro na Validação (Val Loss)', linewidth=2, linestyle='--')
plt.title('Curvas de Aprendizado: Treino vs Validação', fontsize=16)
plt.xlabel('Épocas', fontsize=12)
plt.ylabel('Erro (RMSE)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Avaliação Final
test_loss = model_real.evaluate(X_test_scaled, y_test, verbose=1, batch_size=32)
print(f"Erro Quadrático Médio (MSE) no Teste: {test_loss}")